In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_absolute_error
import xgboost as xgb
import joblib

print("Loading dataset...")
df = pd.read_csv("../resources/crop_dataset_1M.csv")

# -----------------------
# ENCODING
# -----------------------
cat_cols = ["location","soil_type","season","water_source","crop"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

joblib.dump(encoders, "encoders.pkl")

# -----------------------
# FEATURES
# -----------------------
feature_cols = [
    "location","soil_type","season","water_source","near_water",
    "rainfall","temperature","humidity","cost"
]

# -----------------------
# 1️⃣ CROP MODEL
# -----------------------
print("Training crop model...")

X = df[feature_cols]
y = df["crop"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

crop_model = xgb.XGBClassifier(
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    max_depth=8,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    device="cuda"
)

crop_model.fit(X_train, y_train)

preds = crop_model.predict(X_test)
acc = accuracy_score(y_test, preds)
print("Crop model accuracy:", acc)

joblib.dump(crop_model, "crop_model_xgb_gpu.pkl")

# -----------------------
# 2️⃣ PROFIT MODEL
# -----------------------
print("Training profit model...")

feature_cols_profit = feature_cols + ["crop"]

X = df[feature_cols_profit]
y = df["profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

profit_model = xgb.XGBRegressor(
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    max_depth=10,
    n_estimators=500,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    device="cuda"
)

profit_model.fit(X_train, y_train)

preds = profit_model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print("Profit MAE:", mae)

joblib.dump(profit_model, "profit_model_xgb_gpu.pkl")

print("Training complete.")


Loading dataset...
Training crop model...


c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:52:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:52:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:2676: UserWarning: [18:52:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead

Crop model accuracy: 0.08315
Training profit model...


c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:52:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:52:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Profit MAE: 49661.86406009374
Training complete.


c:\Users\Acer\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:2676: UserWarning: [18:53:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

print("\n==============================")
print("CROP MODEL METRICS")
print("==============================")

# Predict
crop_preds = profit_model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, crop_preds)
print("Accuracy:", acc)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_test, crop_preds))

# -----------------------------
# PROFIT MODEL METRICS
# -----------------------------
print("\n==============================")
print("PROFIT MODEL METRICS")
print("==============================")

profit_preds = profit_model.predict(X_test)

mae = mean_absolute_error(y_test, profit_preds)
rmse = np.sqrt(mean_squared_error(y_test, profit_preds))
r2 = r2_score(y_test, profit_preds)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)



CROP MODEL METRICS


ValueError: feature_names mismatch: ['location', 'soil_type', 'season', 'water_source', 'near_water', 'rainfall', 'temperature', 'humidity', 'cost'] ['location', 'soil_type', 'season', 'water_source', 'near_water', 'rainfall', 'temperature', 'humidity', 'cost', 'crop']
training data did not have the following fields: crop